# Démo simulation DSF microscope Bumbleblee (bras radpol)

Toutes les données propres au microscope sont déjà présentes dans le dossier simu_dsf_bubmble_rad. 

In [1]:
%matplotlib qt
from simu_dsf_bumble_rad import *
from numpy.random import normal, poisson
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [2]:
#Détails microscope

lambd = 617 #longueur d'onde de travailen nm
n1 = 1.52 #indice du verre et de l'huile de contact
n2 = 1.33 #indice de l'échantillon biologique est celui de l'eau
h = 6.626*10**(-34) # Planck 
c = 299792458 #vitesse de la lumière

#Nikon Eclipse Ti2-E
f_tube = 200
f_obj = 2
mag_obj = 100
NA = 1.4

#Système relai de lentilles
mag_total = 37.5

#Hamamatsu CMOS
l_pixel = 4.6 #taille pixel - µm
largeur_pixel = 4096 #largeur du capteur - pixel
hauteur_pixel = 2304 #hauteur du capteur - pixel
pixel_sur_camera = l_pixel/mag_total # µm/pixel




In [3]:
N = 80 #discretization de la BFP, numériquement optimisé

#On peut également chercher à obtenir une taille de visualisation finale de la même taille que la FOV réelle en caméra, pour cela: 
#N_tot = max(largeur_pixel, hauteur_pixel) // pixel_sur_camera #On veut que l'image finale soit équivalente à la FOV
#N = padding_depuis_FOV(r_cut, Ntot, lambd, f_tube, f_obj, mag_obj, mag_total, l_pixel, n1)

In [4]:
x, y, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut= vectorial_BFP(N, NA, n1,n2)

In [5]:
#Padding
Npad = padding_depuis_BFP(r_cut, N, lambd, f_tube, f_obj, mag_obj, mag_total, l_pixel, n1)
th1 = pad(th1, Npad)
phi = pad(phi, Npad)
Ex0 = pad(Ex0, Npad)
Ex1 = pad(Ex1, Npad)
Ex2 = pad(Ex2, Npad)
Ey0 = pad(Ey0, Npad)
Ey1 = pad(Ey1, Npad)
Ey2 = pad(Ey2, Npad)

In [6]:
#Définitions des deux bras du radpol 
polar_projections = np.array(['radphi',0])

In [51]:
#Paramètres en µm pour la position, en degrés pour l'orientation, des différents émetteurs
xp = np.array([-9,9])
yp = np.array([0,0])
zp = np.array([1, 1])  #distance au plan focal ,défini à partir de l'interface
d = np.array([-1.3,-1.8]) #distance entre la lamelle et les plans focaux, par convention négatif. défocus de 500 nm sur le second bras
rho = np.array([45, 45])
eta = np.array([30, 30])
delta = np.array([180, 10]) #émetteur isotrope et non istrope donc
N_photons = np.array([5000, 5000]) #Nbre de photons par dipôle

#Paramètres ensuite pour la visualisation
img_shape = th1.shape[-2:]          # (ny, nx)
cx, cy = img_shape[1] / 2, img_shape[0] / 2  # centre en pixels
xp_px = cx + xp / pixel_sur_camera
yp_px = cy + yp / pixel_sur_camera


In [7]:
# 10 émetteurs espacés de 1µm en x et y, même z, rho, delta, N_photons
xp =  np.linspace(-5, 5, 11)#np.ones(10) * 0  #       # [-5, -3.89, -2.78, ..., 5] µm
yp =  np.ones(11) * 0  #      # [-5, -3.89, -2.78, ..., 5] µm
zp = np.ones(11) * 1.0            # z = 1 µm pour tous
d = np.array([-1.3,-1.8])         
rho =  np.ones(11) * 10             # rho = 45° pour tous
eta = np.ones(11) * 90    # eta distribué uniformément entre 0° et 90°
delta = np.ones(11) * 180.0     #np.linspace(10, 180, 10)           # delta = 10 pour tous
N_photons = np.ones(11, dtype=int) * 5000  # 5000 photons par dipôle

#Paramètres ensuite pour la visualisation
img_shape = th1.shape[-2:]          # (ny, nx)
cx, cy = img_shape[1] / 2, img_shape[0] / 2  # centre en pixels
xp_px = cx + xp / pixel_sur_camera
yp_px = cy + yp / pixel_sur_camera

In [8]:
#Obtention de la base de PSF, la matrice M
M = compute_M(xp,yp,zp,d,th1,phi,Ex0,Ex1,Ex2,Ey0,Ey1,Ey2, n1,n2, pixel_sur_camera, polar_projections = polar_projections, lambd=lambd)

In [9]:
#Maintenant, ajoutons l'effet dipôle au lieu de simple émetteur et obtenons le plan image
#Att, la taille affichée constitue celle du padding. la vraie fov est de 4096x2304 pixels
psf = PSF(rho,eta,delta,d, M,N_photons)

In [11]:
fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.05], wspace=0.3, hspace=0.4)

psf_sum = psf.sum(axis=0)
vmin = psf_sum.min()
vmax = psf_sum.max()

#Dans le cas où on visualise toute la FOV, faut cropper le côté pas carré
'''Nx = int(np.ceil(FoV_x / pixel_sur_camera))
Ny = int(np.ceil(FoV_y / pixel_sur_camera))

cx, cy = N_total // 2, N_total // 2

psf_crop = psf[cy - Ny//2 : cy + Ny//2,
               cx - Nx//2 : cx + Nx//2]'''

titles = [[f'PSF d={d[i]}, {("rad" if p == "radphi" else f"{int(p)}°")}', 
           f'PSF d={d[i]}, {("phi" if p == "radphi" else f"{int(p)+90}°")}']
          for i, p in enumerate(polar_projections)]

axes = [[fig.add_subplot(gs[i, j]) for j in range(2)] for i in range(2)]


for i in range(2):
    for j in range(2):
        im = axes[i][j].imshow(psf_sum[i, j, :, :], vmin=vmin, vmax=vmax, origin='lower')
        axes[i][j].set_title(titles[i][j], pad=8)
        axes[i][j].plot(xp_px, yp_px, 'r+', markersize=2, markeredgewidth=1.5)

cax = fig.add_subplot(gs[:, 2])
fig.colorbar(im, cax=cax)

info_lines = [
    f"Emitter {k+1}: xp={xp[k]} µm, yp={yp[k]} µm, zp={zp[k]} µm,"
    f"ρ={rho[k]}°, η={eta[k]}°, δ={delta[k]}°, N_photons={N_photons[k]}"
    for k in range(len(xp))
]
fig.text(0.5, 0.01, '\n'.join(info_lines), ha='center', va='bottom',
         fontsize=11, family='monospace')

plt.subplots_adjust(bottom=0.25)
plt.show()

In [10]:
plt.close('all')

half = 15 #centre de psf qu'on attend des données du microscope
n_emetteurs = len(xp)
param_label = "xp"
param_unite = "µm"
param_values = xp

row_labels = ['rad', 'phi', '0°', '90°']
psf_rows = [
    psf[:, 0, 0, :, :],
    psf[:, 0, 1, :, :],
    psf[:, 1, 0, :, :],
    psf[:, 1, 1, :, :],
]

n_rows = 4
fig = plt.figure(figsize=(n_emetteurs * 1.8, n_rows * 1.8 + 0.6))

gs = fig.add_gridspec(n_rows + 1, n_emetteurs + 1,  # +1 colonne pour colorbar
                      width_ratios=[1] * n_emetteurs + [0.05],
                      height_ratios=[0.3] + [1] * n_rows,
                      hspace=0.05, wspace=0.05)

# --- Flèche ---
ax_arrow = fig.add_subplot(gs[0, :-1])
ax_arrow.set_xlim(0, 1)
ax_arrow.set_ylim(0, 1)
ax_arrow.axis('off')
ax_arrow.annotate('', xy=(0.95, 0.5), xytext=(0.05, 0.5),
                  arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax_arrow.text(0.5, 0.85, f'{param_label} : {param_values[0]:.0f} {param_unite} → {param_values[-1]:.0f}{param_unite}',
              ha='center', va='center', fontsize=10)

# --- Calcul vmin/vmax commun ---
img_shape = psf_rows[0].shape[-2:]
cx_im, cy_im = img_shape[1] / 2, img_shape[0] / 2

vmin = min(r.min() for r in psf_rows)
vmax = max(r.max() for r in psf_rows)

# --- Crops ---
im_ref = None
for row in range(n_rows):
    for col in range(n_emetteurs):
        ax = fig.add_subplot(gs[row + 1, col])

        xc = int(round(cy_im + xp[col] / pixel_sur_camera))  # attention x→col
        yc = int(round(cx_im + yp[col] / pixel_sur_camera))  # y→row

        x0, x1 = xc - half, xc + half
        y0, y1 = yc - half, yc + half

        # Clamp aux bords
        x0c, x1c = max(0, x0), min(img_shape[1], x1)
        y0c, y1c = max(0, y0), min(img_shape[0], y1)
        crop = psf_rows[row][col, y0c:y1c, x0c:x1c]

        im_ref = ax.imshow(crop, origin='lower', vmin=vmin, vmax=vmax, aspect='equal')
        ax.set_xticks([])
        ax.set_yticks([])

        # Croix rouge — coordonnées dans le repère du crop
        xp_px_crop = cx_im + xp[col] / pixel_sur_camera - x0c
        yp_px_crop = cy_im + yp[col] / pixel_sur_camera - y0c
        ax.plot(xp_px_crop, yp_px_crop, 'r+', markersize=6, markeredgewidth=1.0)

        # Label ligne à gauche
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=9, rotation=0,
                          labelpad=30, va='center')

        # Valeur param en dessous dernière ligne
        if row == n_rows - 1:
            ax.set_xlabel(f'{param_values[col]:.0f}{param_unite}', fontsize=8)

# --- Colorbar commune ---
cax = fig.add_subplot(gs[1:, -1])
fig.colorbar(im_ref, cax=cax)

# Dictionnaire de tous les paramètres
all_params = {
    'xp': (xp, 'µm'),
    'yp': (yp, 'µm'),
    'zp': (zp, 'µm'),
    'ρ':  (rho, '°'),
    'η':  (eta, '°'),
    'δ':  (delta, ''),
    'N_photons': (N_photons, ''),
}

# Construit le titre : paramètre variable omis, les autres à l'indice 0
title_parts = []
for name, (values, unit) in all_params.items():
    if name == param_label:
        continue  # on skip le paramètre qui varie
    title_parts.append(f'{name}={values[0]}{unit}')

fig.suptitle('Emitters: ' + ', '.join(title_parts)+f'NA={NA}', fontsize=10)
plt.show()

In [42]:
np.sqrt(0.12)

np.float64(0.34641016151377546)